In [4]:
import re, json, itertools
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen3-14B"            # or "Qwen/Qwen3-4B" / "Qwen/Qwen3-8B" for smaller GPUs
EMB_NAME = "Qwen/Qwen3-Embedding-0.6B"
CATEGORY_CLAUSE = " in the category 'beauty products'"
SEED = 42
torch.manual_seed(SEED)

/home/eduardo/ghc2f/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_parquet('../../dataset/all_beauty.parquet')

In [ ]:
users = df.groupby('userId').count()
items = df.groupby('itemId').count()
list_user_review = {}
list_item_review = {}

In [ ]:
# userId
id = 628543
rev = df[df['userId'] == id]['text'].to_list()
list_user_review[id] = rev

In [ ]:
list_user_review

In [ ]:
items[items['rating']>3]['rating']

In [ ]:
# itemId
id = 76871
rev = df[df['itemId'] == id]['text'].to_list()
list_item_review[id] = rev

In [9]:
list_item_review

NameError: name 'list_item_review' is not defined

In [ ]:
list_user_review# Reviews each user wrote in the past (about OTHER items) -> user profile
user_reviews = {
    "u1_commuter": [
        "I take a 90-minute train every day and these block out the engine rumble almost completely. Battery lasted the whole week of commuting. Only wish the case were smaller.",
        "Returned these: the noise cancelling is weak, I could still hear every announcement on the train, and they got uncomfortable after an hour.",
        "Arrived two days late and the box was crushed.",
        "Folds flat and fits in my bag, which matters since I carry it everywhere. Cancellation is decent on planes.",
    ],
    "u2_runner": [
        "Finally earbuds that stay in during sprints. The hooks keep them locked even when I'm drenched in sweat. Sound is fine, not that I care much.",
        "Kept falling out of my ears on every run, useless for training.",
        "Great!!! 5 stars",
        "Survived a rainy half-marathon without issues. Controls are easy to hit with sweaty fingers.",
    ],
    "u3_audiophile": [
        "Way too much bass, it drowns out the vocals and the mids sound muddy. Not for anyone who listens to classical or jazz.",
        "Beautifully neutral and detailed; I can hear the room reverb in old jazz recordings. Needs a decent amp to shine.",
        "Seller was very helpful answering my questions.",
        "I prefer wired over bluetooth; the compression on this one is obvious on acoustic recordings.",
    ],
    "u4_gamer": [
        "My teammates say the mic is crystal clear and there's no noticeable delay in shooters.",
        "Bluetooth lag made it unplayable for competitive games; fine for music I guess.",
        "Bought as a gift, haven't tried it yet.",
        "Comfortable for long evening sessions and the mic mute button is easy to find.",
    ],
}

# Reviews written by other users about candidate items -> item profile
item_reviews = {
    "i1_travel_anc": [
        "The noise cancelling is superb on flights, and 30+ hours of battery means I rarely charge.",
        "Folds compactly; comfortable for long trips though a bit warm.",
        "Bass-heavy sound, not accurate, but great for blocking out the subway.",
    ],
    "i2_sport_earbuds": [
        "Ear hooks keep them in place during runs and HIIT, and they're sweatproof.",
        "Sound is average and there's no noise cancelling, but they never fall out.",
        "Package arrived quickly.",
    ],
    "i3_studio_openback": [
        "Flat, detailed sound with a wide soundstage; ideal for acoustic and classical music.",
        "Wired only and leaks sound, so useless on public transport.",
        "Benefits from a headphone amp; with one they sound fantastic.",
    ],
    "i4_gaming_headset": [
        "Low-latency wireless dongle, no delay in competitive shooters.",
        "The boom mic is clear, my friends hear me perfectly on Discord.",
        "Heavy on the head after several hours and the sound is just okay for music.",
    ],
}

users, items = list(user_reviews), list(item_reviews)

In [5]:
tok = AutoTokenizer.from_pretrained(LLM_NAME)
tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype="auto", device_map="auto")
llm.eval()

def generate(prompts, max_new_tokens=300, batch_size=4):
    """Batched generation with thinking disabled. Qwen recommends sampling
    (temp 0.7, top_p 0.8, top_k 20) in non-thinking mode rather than greedy."""
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        texts = [
            tok.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True, enable_thinking=False,
            )
            for p in batch
        ]
        enc = tok(texts, return_tensors="pt", padding=True).to(llm.device)
        with torch.no_grad():
            gen = llm.generate(
                **enc, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
            )
        for g in gen[:, enc["input_ids"].shape[1]:]:
            text = tok.decode(g, skip_special_tokens=True)
            outputs.append(re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip())
    return outputs

Loading checkpoint shards: 100%|██████████| 8/8 [00:04<00:00,  1.94it/s]


In [ ]:
USEFUL_TEMPLATE = """Below is a review a user wrote about an item{category_clause}. Decide whether \
the review text contains information that would be USEFUL to a review-aware recommender \
system, i.e., information that goes beyond what the numeric rating alone already conveys.

A review is USEFUL if it contains at least one of the following:
- Item aspects: specific properties of the item and how the user evaluated them \
(e.g., quality, size/fit, durability, taste, performance, plot, usability, price-value).
- Reasons: why the user liked or disliked the item, not just that they did.
- User preferences or context: the author's tastes, needs, usage situation, or \
expertise that help characterize who the item suits.
- Comparisons: explicit comparisons to other items, brands, versions, or expectations.

A review is NOT USEFUL if its content is only:
- Generic sentiment with no specifics (e.g., "Great!", "Loved it", "5 stars").
- Issues unrelated to the item itself: shipping, delivery, packaging, seller, \
customer service, or platform problems.
- Statements that the item was not yet used or was a gift not yet opened.
- Off-topic, spam, promotional, or unintelligible text.

Guidelines:
- Judge informativeness, not sentiment, writing quality, or length.
- A review mixing useless content with at least one genuine item aspect or preference is USEFUL.
- Do not assume any label in advance.

Review:
\"\"\"{review}\"\"\"

Answer in exactly this format:
Aspects: <comma-separated item aspects or preferences mentioned, or "none">
Useful: <yes | no>"""

def parse_useful(text):
    m = re.search(r"useful\s*:\s*(yes|no)", text, flags=re.I)
    return m.group(1).lower() == "yes" if m else None   # None = unparseable, inspect it

rows = []
for kind, source in [("user", list_user_review), ("item", list_item_review)]:
    for owner, revs in source.items():
        for r in revs:
            rows.append({"kind": kind, "owner": owner, "review": r})
df = pd.DataFrame(rows)

raw = generate([USEFUL_TEMPLATE.format(category_clause=CATEGORY_CLAUSE, review=r) for r in df.review],
               max_new_tokens=80)
df["llm_output"] = raw
df["useful"] = [parse_useful(t) for t in raw]
print("Unparseable outputs:", df.useful.isna().sum())
pd.set_option("display.max_colwidth", 90)
df[["owner", "review", "useful"]]

In [31]:
df= pd.read_csv('useful_reviews.csv')

In [64]:
MAX_REVIEWS = 30     # cap per user/item to keep prompts short (adjust to your data)
MAX_CHARS = 1000     # truncate very long reviews
CATEGORY_CLAUSE = " in the category 'Beauty / Skin care'"   # adapt, or "" for none

USER_PROFILE_TEMPLATE = """Below are reviews written by one user about different items {category_clause}. \
Summarize this user's preferences as expressed in the reviews.

Step 1 - Filter: Some reviews may be uninformative. Completely ignore reviews whose content \
is only generic praise or complaints (e.g., "Great!", "Love it", "Terrible"), shipping, \
delivery, packaging, seller, customer service, store prices, items not yet used or given \
as gifts, or off-topic text. Base the profile ONLY on reviews that describe item aspects, \
the user's reasons for liking or disliking something, their needs, or their usage context.

Step 2 - Summarize, following these rules:
- Use ONLY information stated in the informative reviews. Do not infer demographics, \
personality, or preferences that are not expressed.
- Describe the user, not any single item. Prefer aspects that appear in several reviews.
- Include personal context only when it explains what the user needs from items \
(e.g., skin type, a condition, a hobby, experience level).
- Be specific and keep the user's own distinctive terms; avoid generic phrases that \
would describe any user (e.g., "values quality").
- If no review is informative, write "none" in every field.

Reviews:
{reviews}

Answer in exactly this format:
Likes: <aspects the user values, comma-separated>
Dislikes: <aspects the user criticizes, comma-separated>
Context: <usage situations, needs, or expertise mentioned>
Summary: <one or two sentences describing this user's preferences>"""

ITEM_PROFILE_TEMPLATE = """Below are reviews written by different users about one item {category_clause}. \
Summarize the characteristics of this item as described in the reviews.

Step 1 - Filter: Some reviews may be uninformative. Completely ignore reviews whose content \
is only generic praise or complaints (e.g., "Great!", "Love it", "Terrible"), shipping, \
delivery, packaging condition on arrival, seller, customer service, store prices, items \
not yet used, or off-topic text. Base the profile ONLY on reviews that describe the item \
or how it worked for the reviewer.

Step 2 - Summarize, following these rules:
- Use ONLY information stated in the informative reviews. Do not add product knowledge \
from elsewhere.
- Prefer aspects mentioned by several reviewers; mention disagreements briefly \
(e.g., "scent: liked by some, disliked by others").
- Be specific; avoid generic phrases that would describe any item.
- If no review is informative, write "none" in every field.

Reviews:
{reviews}

Answer in exactly this format:
Strengths: <aspects reviewers praise, comma-separated>
Weaknesses: <aspects reviewers criticize, comma-separated>
Suits: <users or usage situations the item fits, according to the reviews>
Summary: <one or two sentences describing the item>"""

# ---------- data preparation ----------
df = df.drop(columns=["Unnamed: 0", "useful", "llm_output"], errors="ignore")
df["owner"] = df["owner"].astype(str)
df["review"] = df["review"].fillna("").astype(str).str.strip()
df = df[df.review != ""]

users = df.loc[df.kind == "user", "owner"].unique().tolist()
items = df.loc[df.kind == "item", "owner"].unique().tolist()
print(f"{len(users)} users, {len(items)} items")

def reviews_for(kind, owner):
    revs = df.loc[(df.kind == kind) & (df.owner == owner), "review"]
    revs = list(dict.fromkeys(revs))                 # drop exact duplicates, keep order
    return [r[:MAX_CHARS] for r in revs[:MAX_REVIEWS]]

def fmt(revs):
    return "\n".join(f"{n}. {r}" for n, r in enumerate(revs, 1))

def parse_fields(text, keys):
    out = {}
    for k in keys:
        m = re.search(rf"^\s*\**{k}\**\s*:\s*(.+)$", text, flags=re.I | re.M)
        out[k] = m.group(1).strip() if m else "none"
    out["raw"] = text
    return out

def generate_sorted(prompts, **kwargs):
    """Generate in order of prompt length (less padding per batch), return in original order."""
    order = sorted(range(len(prompts)), key=lambda k: len(prompts[k]))
    out = generate([prompts[k] for k in order], **kwargs)
    result = [None] * len(prompts)
    for pos, k in enumerate(order):
        result[k] = out[pos]
    return result

USER_KEYS = ["Likes", "Dislikes", "Context", "Summary"]
ITEM_KEYS = ["Strengths", "Weaknesses", "Suits", "Summary"]

# ---------- profile generation ----------
profiles = {}   # profiles[(kind, owner)] = dict of fields

u_prompts = [USER_PROFILE_TEMPLATE.format(category_clause=CATEGORY_CLAUSE,
             reviews=fmt(reviews_for("user", u))) for u in users]
i_prompts = [ITEM_PROFILE_TEMPLATE.format(category_clause=CATEGORY_CLAUSE,
             reviews=fmt(reviews_for("item", i))) for i in items]

u_out = generate_sorted(u_prompts, max_new_tokens=300, batch_size=8)
i_out = generate_sorted(i_prompts, max_new_tokens=300, batch_size=8)

for u, t in zip(users, u_out):
    profiles[("user", u)] = parse_fields(t, USER_KEYS)
for i, t in zip(items, i_out):
    profiles[("item", i)] = parse_fields(t, ITEM_KEYS)

# ---------- inspect and save ----------
for key in [("user", u) for u in users[:3]] + [("item", i) for i in items[:3]]:
    print(f"=== {key[0]} {key[1]} ===")
    print(profiles[key]["raw"], "\n")

prof_df = pd.DataFrame([{"kind": k, "owner": o, **{f: v for f, v in p.items()}}
                        for (k, o), p in profiles.items()])
prof_df.to_csv("profiles1.csv", index=False)
print("Profiles with all fields 'none':",
      (prof_df[["Summary"]].apply(lambda s: s.str.lower().eq("none")).sum()).item())

2 users, 2 items
=== user 340457 ===
Likes: moisture retention, anti-aging properties, ease of application, fragrance (specific scents like cucumber and aloe, lavender), products for sensitive skin, multi-purpose products, travel-friendly packaging, skin repair and healing, natural and gentle ingredients
Dislikes: fragrances that are too strong, product leakage, not enough product quantity, poor packaging protection, product not matching skin tone, tangling of hair, not enough ink in kits, products that do not last all day
Context: has eczema and psoriasis, has grandchildren with skin issues, uses products for both skin and hair care, values natural and gentle ingredients, uses products for travel, has experience with tattoos and hair extensions, uses products for anti-wrinkling and skin repair
Summary: The user prefers skincare and haircare products that offer moisture retention, anti-aging benefits, are gentle on sensitive skin, and have natural ingredients. They value ease of applic

In [65]:
prof1 = pd.read_csv("profiles.csv", dtype={"owner": str}, keep_default_na=False)

# ---------- inspect ----------
def show(kind, n=3):
    for _, row in prof1[prof1.kind == kind].head(n).iterrows():
        print(f"=== {kind} {row.owner} ===")
        print(row["raw"], "\n")

show("user")
show("item")

# look up one specific profile by id
by_key = prof1.set_index(["kind", "owner"])
print(by_key.loc[("user", "340457"), ["Likes", "Dislikes", "Context", "Summary"]])

# profiles where the model found no informative review
print("Profiles with Summary 'none':", prof1["Summary"].str.lower().eq("none").sum())

=== user 340457 ===
Likes: moisturizing products, anti-aging and wrinkle prevention, products with natural ingredients, convenience in travel-sized products, products that help with skin healing and repair, products that can be used on sensitive skin, products with pleasant scents, foundation with sunscreen, products that can be used on hair and skin
Dislikes: fragranced products, leaking packaging, products that do not match skin tone, products that do not last long, products that do not provide sufficient coverage for age spots, products that are not suitable for thick hair, products that cause tangling
Context: user has eczema and psoriasis, has grandchildren with skin issues, uses products for both skin and hair care, prefers products that are gentle on sensitive skin, uses products for anti-aging, uses products for travel
Summary: The user values moisturizing and anti-aging products that are gentle on sensitive skin, prefers items with natural ingredients and pleasant scents, and 

In [ ]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer(EMB_NAME)

def embed(texts):
    return emb_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)

def mean_embed(texts):
    v = embed(texts).mean(axis=0)
    return v / np.linalg.norm(v)

def field_vec(profile, key):
    val = profile.get(key, "none")
    if not val or val.strip().lower() in {"none", "n/a", "-"}:
        return np.zeros(emb_model.get_sentence_embedding_dimension())
    return embed([val])[0]

def profile_text(p, keys):
    return "\n".join(f"{k}: {p[k]}" for k in keys)

In [ ]:
def matrix(U, I):
    return pd.DataFrame(np.vstack(U) @ np.vstack(I).T, index=users, columns=items)

scores, user_vecs = {}, {}

# A/B. Raw reviews, mean-pooled
for setting, only_useful in [("all", False), ("useful", True)]:
    U = [mean_embed(reviews_for(u, only_useful)) for u in users]
    I = [mean_embed(reviews_for(i, only_useful)) for i in items]
    scores[f"raw_{setting}"], user_vecs[f"raw_{setting}"] = matrix(U, I), U

# C/D. LLM summaries (full structured profile embedded as one text)
for setting in ["all", "useful"]:
    U = [embed([profile_text(profiles[(setting, u)], USER_KEYS)])[0] for u in users]
    I = [embed([profile_text(profiles[(setting, i)], ITEM_KEYS)])[0] for i in items]
    scores[f"summary_{setting}"], user_vecs[f"summary_{setting}"] = matrix(U, I), U

# E. Signed aspect matching (useful profiles):
#    score = sim(Likes, Strengths) - sim(Likes, Weaknesses) - sim(Dislikes, Strengths)
S = np.zeros((len(users), len(items)))
for a, u in enumerate(users):
    L, D = field_vec(profiles[("useful", u)], "Likes"), field_vec(profiles[("useful", u)], "Dislikes")
    for b, i in enumerate(items):
        St, W = field_vec(profiles[("useful", i)], "Strengths"), field_vec(profiles[("useful", i)], "Weaknesses")
        S[a, b] = L @ St - L @ W - D @ St
scores["aspect_signed_useful"] = pd.DataFrame(S, index=users, columns=items)

In [ ]:
def evaluate(M):
    A = M.values
    n = len(A)
    hit1 = np.mean([A[k].argmax() == k for k in range(n)])
    margin = np.mean([A[k, k] - np.delete(A[k], k).mean() for k in range(n)])
    return hit1, margin

def homogeneity(vecs):
    V = np.vstack(vecs)
    return np.mean([V[a] @ V[b] for a, b in itertools.combinations(range(len(V)), 2)])

summary = pd.DataFrame(
    [(name, *evaluate(M), homogeneity(user_vecs[name]) if name in user_vecs else np.nan)
     for name, M in scores.items()],
    columns=["representation", "hit@1", "margin", "user_homogeneity"],
).set_index("representation")
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(scores), figsize=(4.2 * len(scores), 4))
for ax, (name, M) in zip(axes, scores.items()):
    im = ax.imshow(M.values, cmap="viridis")
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(len(items)), [i.split("_", 1)[1] for i in items], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(users)), [u.split("_", 1)[1] for u in users], fontsize=8)
    for a in range(len(users)):
        for b in range(len(items)):
            ax.text(b, a, f"{M.values[a, b]:.2f}", ha="center", va="center", color="w", fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
def justify(u, i, setting="useful"):
    pu, pi = profiles[(setting, u)], profiles[(setting, i)]
    print(f"User {u}\n  Likes: {pu['Likes']}\n  Dislikes: {pu['Dislikes']}")
    print(f"Item {i}\n  Strengths: {pi['Strengths']}\n  Weaknesses: {pi['Weaknesses']}")
    # evidence: most similar pair (user review sentence, item review sentence)
    ur, ir = reviews_for(u, True), reviews_for(i, True)
    sim = embed(ur) @ embed(ir).T
    a, b = np.unravel_index(sim.argmax(), sim.shape)
    print(f"\nEvidence (cos={sim[a, b]:.2f}):\n  user said: {ur[a]}\n  reviewers said: {ir[b]}")

justify("u3_audiophile", "i3_studio_openback")
print("\n" + "-" * 80 + "\n")
justify("u3_audiophile", "i1_travel_anc")   # expected mismatch: bass-heavy sound

In [29]:
df_aspects = pd.read_csv('../aspects/extracted_aspects/all_beauty_absa_test_aspects.csv')

In [37]:
df_aspects[(df_aspects['userId'] == 340457]) & (df_aspects['sentiment'] == 'positive'])

,review_idx,userId,itemId,raw_user,raw_item,cluster,aspect,sentiment,polarity,mentions,confidence,pct_classifier,prob_positive,prob_negative,prob_neutral
10003,1741,340457,52159,340457,52159,62130,oxygenceutical,positive,0.942354,1,0.945679,1.0,0.945679,0.003325,0.050996
10004,1741,340457,52159,340457,52159,78685,skin,positive,0.592835,1,0.602345,1.0,0.602345,0.009510,0.388144
10005,1742,340457,74660,340457,74660,27856,eczema,positive,0.743547,1,0.787229,1.0,0.787229,0.043681,0.169090
10006,1742,340457,74660,340457,74660,68432,psoriasis,positive,0.497064,1,0.564744,1.0,0.564744,0.067680,0.367576
10011,1743,340457,75649,340457,75649,74689,scalp,positive,0.994649,1,0.996383,1.0,0.996383,0.001734,0.001883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10680,1901,340457,11540,340457,11540,31525,facial moisturizer,positive,0.996169,1,0.997032,1.0,0.997032,0.000862,0.002106
10682,1901,340457,11540,340457,11540,55320,moisturizer,positive,0.989919,1,0.991142,1.0,0.991142,0.001223,0.007634
10684,1902,340457,4875,340457,4875,34011,foot,positive,0.985779,1,0.990760,1.0,0.990760,0.004980,0.004260
10685,1902,340457,4875,340457,4875,34032,foot buffer,positive,0.997370,1,0.998068,1.0,0.998068,0.000698,0.001234


In [14]:
CATEGORY_CLAUSE = "in the category {category} "
CATEGORY_CLAUSE.format(category="teste")


'in the category teste '

In [7]:
ITEM_PROFILE_TEMPLATE = """Below are reviews written by different users about one item{category_clause}. \
Summarize the characteristics of this item as described in the reviews.

Step 1 - Filter: Some reviews may be uninformative. Completely ignore reviews whose content \
is only generic praise or complaints (e.g., "Great!", "Love it", "Terrible"), shipping, \
delivery, packaging condition on arrival, seller, customer service, store prices, items \
not yet used, or off-topic text. Base the profile ONLY on reviews that describe the item \
or how it worked for the reviewer.

Step 2 - Summarize, following these rules:
- Use ONLY information stated in the informative reviews. Do not add product knowledge \
from elsewhere.
- Prefer aspects mentioned by several reviewers; mention disagreements briefly \
(e.g., "scent: liked by some, disliked by others").
- Be specific; avoid generic phrases that would describe any item.
- If no review is informative, write "none" in every field.

Reviews:
{reviews}

Answer in exactly this format:
Strengths: <aspects reviewers praise, comma-separated>
Weaknesses: <aspects reviewers criticize, comma-separated>
Suits: <users or usage situations the item fits, according to the reviews>
Summary: <one or two sentences describing the item>"""

In [11]:
ITEM_PROFILE_TEMPLATE.format(category_clause='Skin', reviews={'this is a review'})

'Below are reviews written by different users about one itemSkin. Summarize the characteristics of this item as described in the reviews.\n\nStep 1 - Filter: Some reviews may be uninformative. Completely ignore reviews whose content is only generic praise or complaints (e.g., "Great!", "Love it", "Terrible"), shipping, delivery, packaging condition on arrival, seller, customer service, store prices, items not yet used, or off-topic text. Base the profile ONLY on reviews that describe the item or how it worked for the reviewer.\n\nStep 2 - Summarize, following these rules:\n- Use ONLY information stated in the informative reviews. Do not add product knowledge from elsewhere.\n- Prefer aspects mentioned by several reviewers; mention disagreements briefly (e.g., "scent: liked by some, disliked by others").\n- Be specific; avoid generic phrases that would describe any item.\n- If no review is informative, write "none" in every field.\n\nReviews:\n{\'this is a review\'}\n\nAnswer in exactly

In [159]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
li = lu = LabelEncoder()

In [181]:
dataset = 'all_beauty'
df = pd.read_parquet(f'../../dataset/{dataset}.parquet')

In [182]:
df

,rating,title,text,asin,parent_asin,user_id,timestamp,userId,itemId
0,5,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,B08P2DZB4X,B08P2DZB4X,AFSKPY37N3C43SOI5IEXEK5JSIYA,2021-07-27 13:04:04.559,3620,15571
1,5,Nice shampoo for the money,I get Keratin treatments at the salon at least...,B086QY6T7N,B086QY6T7N,AFSKPY37N3C43SOI5IEXEK5JSIYA,2021-07-18 13:21:51.145,3620,12811
2,3,Not what I thought I would be getting,I was very disappointed when I got this facial...,B08DHTJ25J,B08DHTJ25J,AFSKPY37N3C43SOI5IEXEK5JSIYA,2021-07-13 21:20:45.370,3620,14169
3,5,A little goes a long way!,This is a really nice moisturizing lotion. It ...,B07RBSLNFR,B07RBSLNFR,AFSKPY37N3C43SOI5IEXEK5JSIYA,2021-05-16 17:00:30.697,3620,10273
4,3,Just ok,I try to get Keratin treatments every 3 months...,B07SLFWZKN,B07SLFWZKN,AFSKPY37N3C43SOI5IEXEK5JSIYA,2021-04-29 23:05:01.209,3620,10540
...,...,...,...,...,...,...,...,...,...
35619,5,Water waves on swim !!,[[VIDEOID:1552f5dc4ee822d5d7a90ac9288ba72c]] O...,B07NXQYVMH,B07NXQYVMH,AGCESJO2YAHFW3GKOY2JSS5QC6DA,2019-05-02 16:05:38.288,4645,9756
35620,5,Silk silk is that you ??!!,"Omg,omg, omg!!! Want silky hair? Look no furth...",B071RVQQBD,B071RVQQBD,AGCESJO2YAHFW3GKOY2JSS5QC6DA,2018-09-25 02:04:49.116,4645,6242
35621,5,Classic,Classic black polish with not much more to tel...,B005P4AIZ8,B09P3ZMV55,AGNEYGHRA2MM5MUN6WCZRXDSNJZA,2014-06-22 08:31:40.000,5339,18115
35622,2,Not really what it looks like here,On the picture it looks like milky half transp...,B005P49I7W,B09P3ZMV55,AGNEYGHRA2MM5MUN6WCZRXDSNJZA,2014-06-22 08:29:14.000,5339,18115


In [175]:
df.rename({'text':'review'}, axis=1, inplace=True)

In [177]:
filtered_df = df[df.groupby('userId')['userId'].transform('size') >= 3]

In [178]:
filtered_df['userId'] = lu.fit_transform(filtered_df['userId'])
filtered_df['itemId'] = li.fit_transform(filtered_df['itemId'])
filtered_df.reset_index(inplace = True)
filtered_df.drop('index', axis=1, inplace=True)

In [179]:
filtered_df

,rating,title,review,asin,parent_asin,user_id,timestamp,userId,itemId
0,5,Better Than I Expected,At the price I wasn't expecting much from this...,B09XHLT5X8,B09XHMKZYK,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-05-27 14:10:24.284,79762,113900
1,4,Works Well,My 22 year old son has been expanding his soun...,B0812SGFJW,B0812SGFJW,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2021-01-04 15:49:27.637,79762,95133
2,4,Nicer Than Expected,My 17 year old daughter enjoys playing her uku...,B07C8FB2TC,B0BG3T7TJV,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2018-10-12 21:34:55.219,79762,117335
3,4,Decent Sound Quality,These are decent speakers and better than what...,B071JHJ9RV,B071JHJ9RV,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2018-08-04 23:22:30.320,79762,70998
4,5,Fine Quality,My daughter plays the cello in her high school...,B010CG2WOS,B010CG21H6,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2018-04-13 11:26:06.592,79762,54857
...,...,...,...,...,...,...,...,...,...
774263,4,Mostly playable. A value purchase for the curi...,The little pocket sax I recieved was not quite...,B0821JCWTQ,B0821JCWTQ,AGUOIVE2G3HGW7QN5JB456GH5JQA,2020-06-08 07:43:51.408,58789,95713
774264,5,The classic Smokey with a utilitarian enclosure,"Be careful with the speaker, other than that t...",B003U0JYUE,B003U0JYUE,AGUOIVE2G3HGW7QN5JB456GH5JQA,2020-06-08 06:52:28.466,58789,21249
774265,5,Way better than expected.,Far better than expected. The build quality of...,B072V6N7P8,B072V6N7P8,AGUOIVE2G3HGW7QN5JB456GH5JQA,2018-04-25 09:27:13.333,58789,72282
774266,5,Does its job then some!,Does its job and then some. Stuck this pickup ...,B016APM902,B016APM902,AGUOIVE2G3HGW7QN5JB456GH5JQA,2018-02-25 08:20:07.936,58789,57251


In [180]:
filtered_df.to_parquet(f'../../dataset/{dataset}.parquet', index=False)